In [1]:
import pandas as pd
import numpy as np



In [4]:
df = pd.read_json("../src/tracker/candles_1h.jsonl",lines=True,encoding="utf-8")
df.describe()

,open,high,low,close,volume
count,8750.000000,8750.000000,8750.000000,8750.000000,8750.000000
mean,84958.969683,85214.016403,84685.712786,84952.374719,341.625402
std,18997.061748,19024.514460,18961.407781,18994.346881,380.425270
min,58186.390000,58517.280000,57717.550000,58190.000000,2.744184
25%,67578.802500,67859.757500,67310.995000,67574.687500,128.706014
50%,80022.210000,80249.840000,79721.420000,80016.900000,219.622237
75%,102035.430000,102337.932500,101673.840000,102017.260000,406.789707
max,126099.210000,126296.000000,125319.850000,126099.220000,6239.238189


In [5]:
df.head()

,time,symbol,open,high,low,close,volume
0,2025-08-11T09:00:00+00:00,BTC-USD,121662.15,121703.75,121094.37,121406.25,150.695414
1,2025-08-11T10:00:00+00:00,BTC-USD,121407.36,121428.76,120926.35,121150.11,106.300777
2,2025-08-11T11:00:00+00:00,BTC-USD,121150.10,121252.87,120248.43,120525.00,257.085550
3,2025-08-11T12:00:00+00:00,BTC-USD,120525.00,120650.65,119505.00,119616.96,395.203193
4,2025-08-11T13:00:00+00:00,BTC-USD,119620.83,120313.40,119333.35,119976.39,436.222052


In [7]:
df['target']

0       False
1       False
2       False
3        True
4        True
        ...  
8745    False
8746    False
8747     True
8748    False
8749    False
Name: target, Length: 8750, dtype: bool

In [9]:
df

,time,symbol,open,high,low,close,volume,target
0,2025-08-11T09:00:00+00:00,BTC-USD,121662.15,121703.75,121094.37,121406.25,150.695414,False
1,2025-08-11T10:00:00+00:00,BTC-USD,121407.36,121428.76,120926.35,121150.11,106.300777,False
2,2025-08-11T11:00:00+00:00,BTC-USD,121150.10,121252.87,120248.43,120525.00,257.085550,False
3,2025-08-11T12:00:00+00:00,BTC-USD,120525.00,120650.65,119505.00,119616.96,395.203193,True
4,2025-08-11T13:00:00+00:00,BTC-USD,119620.83,120313.40,119333.35,119976.39,436.222052,True
...,...,...,...,...,...,...,...,...
8745,2026-08-11T04:00:00+00:00,BTC-USD,64074.71,64095.91,63956.58,63960.21,88.774600,False
8746,2026-08-11T05:00:00+00:00,BTC-USD,63960.21,63992.19,63881.73,63926.89,79.994356,False
8747,2026-08-11T06:00:00+00:00,BTC-USD,63926.90,63986.16,63783.21,63897.94,87.796354,True
8748,2026-08-11T07:00:00+00:00,BTC-USD,63897.94,64049.29,63836.55,63999.16,89.758020,False


In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_json("../src/tracker/candles_1h.jsonl", lines=True)
df = df.sort_values("time").reset_index(drop=True)

# 1. target -- did the next hour close higher?
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)

# 2. features -- all ratios/returns, never raw dollars
df["ret_1"]    = df["close"].pct_change(1)
df["ret_6"]    = df["close"].pct_change(6)
df["ma_ratio"] = df["close"] / df["close"].rolling(50).mean()

# 3. drop NaNs ONCE, after every column exists
df = df.dropna()

# 4. allowlist -- only columns we built
feature_cols = ["ret_1", "ret_6", "ma_ratio"]
X = df[feature_cols]
y = df["target"]

# 5. chronological split -- no shuffling on time series
n = int(len(df) * 0.8)
X_train, X_test = X[:n], X[n:]
y_train, y_test = y[:n], y[n:]

# 6. fit
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipeline.fit(X_train, y_train)
proba = pipeline.predict_proba(X_test)[:, 1]

# 7. always compare against the do-nothing baseline
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"train/test    : {len(X_train)} / {len(X_test)}")
print(f"baseline      : {baseline*100:.2f}%")
print(f"accuracy      : {accuracy_score(y_test, proba > 0.5)*100:.2f}%")
print(f"roc auc       : {roc_auc_score(y_test, proba):.4f}")


train/test    : 6960 / 1741
baseline      : 50.89%
accuracy      : 52.67%
roc auc       : 0.5371
